:# Projek Klasifikasi Sampah (Organik vs Anorganik) - Kelompok 3
Projek ini menggunakan pendekatan *Transfer Learning* dengan arsitektur **MobileNetV2** untuk mengklasifikasikan sampah menjadi dua kategori: Organik dan Anorganik.

In [ ]:
import tensorflow as tf

# Memastikan Colab menggunakan akselerasi GPU untuk training yang lebih cepat
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow Version:", tf.__version__)

# Melihat detail spesifikasi GPU yang dialokasikan
!nvidia-smi

In [ ]:
import os
from google.colab import files

# 1. Upload file kaggle.json yang diunduh dari akun Kaggle Anda
print("Silakan upload file kaggle.json:")
files.upload()

# 2. Membuat folder .kaggle di sistem Colab
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Mengunduh dataset dari Kaggle
!kaggle datasets download -d techsash/waste-classification-data

# 4. Mengekstrak file zip dataset
!unzip -q waste-classification-data.zip
print("Dataset berhasil diunduh dan diekstrak!")

## 2. Preprocessing & Augmentasi Data
Pada tahap ini, data training akan diberi efek augmentasi agar model lebih generalis. Sedangkan data validasi tetap dibiarkan murni (hanya di-rescale) agar evaluasi performa model bersifat objektif.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Tentukan lokasi folder TRAIN hasil ekstraksi
train_dir = '/content/DATASET/TRAIN'

# Generator untuk Data Training (Dengan Augmentasi Gambar)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # Memisahkan 20% data untuk validasi
)

# Generator untuk Data Validasi (Murni, Tanpa Augmentasi)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Memuat data dari direktori ke dalam Generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

## 3. Pembangunan Model (Transfer Learning MobileNetV2) & Training
Kami menggunakan *Pre-trained model* MobileNetV2 yang dibekukan (*freeze*), kemudian ditambahkan layer *Global Average Pooling* dan satu *Dense layer* dengan aktivasi Sigmoid untuk klasifikasi biner.

In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. Definisi Arsitektur Model (MobileNetV2)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Membukan bobot dasar komponen MobileNetV2

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# 2. Compile Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 3. Konfigurasi Callbacks
callbacks_list = [
    # Berhenti otomatis jika val_loss tidak turun selama 3 epoch berturut-turut
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),

    # Menyimpan model terbaik langsung ke dalam format legacy .h5
    ModelCheckpoint(filepath='model_final.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# 4. Proses Training Model
history = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks_list
)

## 4. Evaluasi Training (Grafik Akurasi & Loss)
Visualisasi performa model selama epoch berlangsung untuk memantau indikasi *overfitting* atau *underfitting*.

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

plt.figure(figsize=(12, 4))

# Subplot 1: Akurasi
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
plt.title('Grafik Akurasi Model')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Subplot 2: Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Grafik Loss Model')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Menyimpan dan otomatis mengunduh hasil grafik
plt.tight_layout()
plt.savefig("grafik_training.png")
plt.show()

print("Mengunduh hasil grafik...")
files.download("grafik_training.png")

## 5. Uji Coba Prediksi Gambar Baru (Deployment Test)
Unggah file gambar dari komputer lokal untuk melihat seberapa akurat model memprediksi jenis sampah tersebut.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# 1. Upload gambar dari komputer lokal
print("Silakan upload gambar sampah yang ingin diuji (Format: .jpg / .jpeg / .png):")
uploaded = files.upload()

# 2. Memuat model terbaik yang sudah disimpan dalam format .h5
model_terlatih = tf.keras.models.load_model('model_final.h5')

# 3. Proses Prediksi untuk setiap gambar yang di-upload
for img_path in uploaded.keys():

    # Preprocessing gambar agar sesuai ukuran input model (224x224)
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Menambahkan dimensi batch
    img_array = img_array / 255.0                  # Normalisasi piksel

    # Eksekusi Prediksi
    prediction = model_terlatih.predict(img_array)

    # Penentuan Kelas (0 = Organik 'O', 1 = Anorganik/Recyclable 'R')
    if prediction[0][0] < 0.5:
        hasil = "ORGANIK"
        persentase = (1 - prediction[0][0]) * 100
    else:
        hasil = "ANORGANIK"
        persentase = prediction[0][0] * 100

    # Menampilkan Gambar beserta Hasil Tebakan
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Prediksi Kelompok 3: {hasil}\nConfidence: {persentase:.2f}%")
    plt.axis('off')
    plt.show()

    print(f"Detail Nilai Output Raw Sigmoid: {prediction[0][0]:.4f}\n")

# 4. Download file model_final.h5 ke komputer lokal
print("Mengunduh file model_final.h5...")
files.download('model_final.h5')